In [17]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time
import matplotlib.pyplot as plt

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [19]:
DATA_FOLDER = '../Weather_Merged_CSVs'
PLOT_FOLDER = 'LSTM_Results'
os.makedirs(PLOT_FOLDER, exist_ok=True)
n_steps_to_test = [7, 14, 21, 30, 60]
all_crops_summary = []

In [20]:
def create_sequences(data, n_steps, target_col):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data.iloc[i:(i + n_steps)].values)
        y.append(data.iloc[i + n_steps][target_col])
    return np.array(X), np.array(y).reshape(-1, 1)

In [21]:
def calculate_mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    nonzero_mask = actual != 0
    if not np.any(nonzero_mask): return float('inf')
    return np.mean(np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])) * 100

In [22]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # LSTM returns output, and a tuple of (hidden_state, cell_state)
        # We only need the output sequence
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [23]:
csv_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]
for csv_file in csv_files:
    try:
        crop_name = os.path.splitext(csv_file)[0]
        file_path = os.path.join(DATA_FOLDER, csv_file)
        print("\n" + "="*70); print(f"Processing Crop: {crop_name}"); print("="*70)

        df = pd.read_csv(file_path)
        df['Price Date'] = pd.to_datetime(df['Price Date'])
        df.set_index('Price Date', inplace=True); df.sort_index(inplace=True)

        if len(df) < 100:
            print(f"Skipping {crop_name} due to insufficient data ({len(df)} rows)."); continue

        df['day_of_year'] = df.index.dayofyear; df['week_of_year'] = df.index.isocalendar().week.astype(int); df['month'] = df.index.month
        categorical_cols = ['District Name', 'Market Name', 'Commodity', 'Variety', 'Grade']
        for col in categorical_cols:
            if df[col].dtype == 'object': df[col] = LabelEncoder().fit_transform(df[col])
        scaler = MinMaxScaler(feature_range=(0, 1))
        df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
        target_column = 'Modal Price (Rs./Quintal)'

        best_mape_for_crop = float('inf'); best_n_steps_for_crop = -1; best_predictions_for_crop = None; best_actuals_for_crop = None

        for n_steps in n_steps_to_test:
            print(f"\n--- Testing {crop_name} with n_steps = {n_steps} ---")
            if len(df_scaled) <= n_steps: print(f"Skipping n_steps={n_steps} as it's too large."); continue
                
            X, y = create_sequences(df_scaled, n_steps, target_column)
            split = int(0.8 * len(X))
            X_train, X_test, y_train, y_test = X[:split], X[split:], y[:split], y[split:]
            X_train_tensor, y_train_tensor = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
            X_test_tensor, y_test_tensor = torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float()
            train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
            test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)
            
            # --- Use the LSTMModel ---
            model = LSTMModel(X_train.shape[2], 50, 2, 1, 0.2).to(device)
            criterion = nn.MSELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            num_epochs = 50
            for epoch in range(num_epochs):
                model.train()
                for batch_X, batch_y in train_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
            
            model.eval()
            all_predictions = []
            with torch.no_grad():
                for batch_X, _ in test_loader:
                    outputs = model(batch_X.to(device))
                    all_predictions.append(outputs.cpu().numpy())
            predictions = np.concatenate(all_predictions)

            price_col_index = df_scaled.columns.get_loc(target_column)
            dummy_pred = np.zeros((len(predictions), df_scaled.shape[1])); dummy_pred[:, price_col_index] = predictions.flatten()
            inversed_predictions = scaler.inverse_transform(dummy_pred)[:, price_col_index]
            dummy_actual = np.zeros((len(y_test), df_scaled.shape[1])); dummy_actual[:, price_col_index] = y_test.flatten()
            inversed_actual = scaler.inverse_transform(dummy_actual)[:, price_col_index]
            
            mape = calculate_mape(inversed_actual, inversed_predictions)
            print(f"n_steps = {n_steps} | Final Test MAPE = {mape:.2f}%")

            if mape < best_mape_for_crop:
                best_mape_for_crop = mape; best_n_steps_for_crop = n_steps
                best_predictions_for_crop = inversed_predictions; best_actuals_for_crop = inversed_actual

        if best_n_steps_for_crop != -1:
            best_rmse = np.sqrt(mean_squared_error(best_actuals_for_crop, best_predictions_for_crop))
            summary = {
                'Crop': crop_name, 'Best n_steps': best_n_steps_for_crop,
                'Best RMSE': round(best_rmse, 2), 'Loss % (MAPE)': round(best_mape_for_crop, 2),
                'Accuracy %': round(100 - best_mape_for_crop, 2)
            }
            all_crops_summary.append(summary)

            plt.figure(figsize=(14, 7))
            plt.plot(best_actuals_for_crop, color='red', label='Actual Price')
            plt.plot(best_predictions_for_crop, color='blue', label=f'Predicted Price (Best n_steps={best_n_steps_for_crop})')
            plt.title(f'LSTM Best Model for {crop_name} (MAPE: {best_mape_for_crop:.2f}%)')
            plt.xlabel('Time (Test Set)'); plt.ylabel('Price (Rs./Quintal)'); plt.legend()
            plot_filename = os.path.join(PLOT_FOLDER, f"{crop_name}_prediction_plot.png")
            plt.savefig(plot_filename); plt.close()
            print(f"\nSaved best plot for {crop_name} to {plot_filename}")

    except Exception as e:
        print(f"\nCould not process {csv_file}. Error: {e}"); continue


Processing Crop: Bajra-2015-2019

--- Testing Bajra-2015-2019 with n_steps = 7 ---
n_steps = 7 | Final Test MAPE = 17.14%

--- Testing Bajra-2015-2019 with n_steps = 14 ---
n_steps = 14 | Final Test MAPE = 12.38%

--- Testing Bajra-2015-2019 with n_steps = 21 ---
n_steps = 21 | Final Test MAPE = 12.98%

--- Testing Bajra-2015-2019 with n_steps = 30 ---
n_steps = 30 | Final Test MAPE = 19.38%

--- Testing Bajra-2015-2019 with n_steps = 60 ---
n_steps = 60 | Final Test MAPE = 22.64%

Saved best plot for Bajra-2015-2019 to LSTM_Results\Bajra-2015-2019_prediction_plot.png

Processing Crop: Bajra-2019-2022
Skipping Bajra-2019-2022 due to insufficient data (55 rows).

Processing Crop: Bajra-2022-2025

--- Testing Bajra-2022-2025 with n_steps = 7 ---
n_steps = 7 | Final Test MAPE = 16.08%

--- Testing Bajra-2022-2025 with n_steps = 14 ---
n_steps = 14 | Final Test MAPE = 16.50%

--- Testing Bajra-2022-2025 with n_steps = 21 ---
n_steps = 21 | Final Test MAPE = 15.38%

--- Testing Bajra-2022-

In [24]:
summary_df = pd.DataFrame(all_crops_summary)
summary_df.sort_values(by='Accuracy %', ascending=False, inplace=True)
summary_filename = './LSTM_Results/LSTM_Results.csv'
summary_df.to_csv(summary_filename, index=False)
print("\n\n" + "="*70); print(f"Processing complete. Summary saved to '{summary_filename}'"); print("="*70)
print(summary_df)



Processing complete. Summary saved to './LSTM_Results/LSTM_Results.csv'
                   Crop  Best n_steps  Best RMSE  Loss % (MAPE)  Accuracy %
13               Garlic            30     778.82           2.55       97.45
21      Maize-2019-2022            30     210.74           7.61       92.39
35               Rubber            60    1215.21           8.00       92.00
5            Cashewnuts            30    2899.32           9.24       90.76
33         Red_Chillies            21    2395.67           9.41       90.59
30       Ragi-2015-2019            21     239.33           9.41       90.59
32       Ragi-2022-2025            60     500.20           9.92       90.08
26                Onion            30     422.67          10.97       89.03
4   Blackgram-2022-2025            60    1102.70          11.40       88.60
1       Bajra-2022-2025            30     313.33          11.59       88.41
0       Bajra-2015-2019            14     828.05          12.38       87.62
20      Maize-